In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [4]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
528,I have seen previous movies from Cédric Klapis...,negative
653,"One of the funniest, most romantic, and most m...",positive
352,I wasted 35 minutes of my life on this turkey ...,negative
6,This is a good example a film that in spite of...,positive
484,Another attempt by modern Japanese directors t...,positive


In [5]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [6]:
df = normalize_text(df)
df.head()

,review,sentiment
528,seen previous movie cédric klapisch therefore ...,negative
653,one funniest romantic musical movie ever defin...,positive
352,wasted minute life turkey gave up main charact...,negative
6,good example film spite low rating worth watch...,positive
484,another attempt modern japanese director redef...,positive


In [7]:
df['sentiment'].value_counts()

sentiment
negative    250
positive    250
Name: count, dtype: int64

In [8]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
528,seen previous movie cédric klapisch therefore ...,0
653,one funniest romantic musical movie ever defin...,1
352,wasted minute life turkey gave up main charact...,0
6,good example film spite low rating worth watch...,1
484,another attempt modern japanese director redef...,1


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=75)

In [13]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/SANJAY-SRINIVAS226/Capstone-project.mlflow')
dagshub.init(repo_owner='SANJAY-SRINIVAS226', repo_name='Capstone-project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

Accessing as SANJAY-SRINIVAS226

2026-06-04 19:17:52,681 - INFO - Accessing as SANJAY-SRINIVAS226


Initialized MLflow to track repo "SANJAY-SRINIVAS226/Capstone-project"

2026-06-04 19:17:53,827 - INFO - Initialized MLflow to track repo "SANJAY-SRINIVAS226/Capstone-project"


Repository SANJAY-SRINIVAS226/Capstone-project initialized!

2026-06-04 19:17:53,833 - INFO - Repository SANJAY-SRINIVAS226/Capstone-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/bb348505c2eb41e7be1a51850d1a38a7', creation_time=1780576618868, experiment_id='0', last_update_time=1780576618868, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [17]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 1000)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-06-04 19:32:27,095 - INFO - Starting MLflow run...
2026-06-04 19:32:28,694 - INFO - Logging preprocessing parameters...
2026-06-04 19:32:29,977 - INFO - Initializing Logistic Regression model...
2026-06-04 19:32:29,980 - INFO - Fitting the model...
2026-06-04 19:32:30,019 - INFO - Model training complete.
2026-06-04 19:32:30,019 - INFO - Logging model parameters...
2026-06-04 19:32:30,452 - INFO - Making predictions...
2026-06-04 19:32:30,456 - INFO - Calculating evaluation metrics...
2026-06-04 19:32:30,480 - INFO - Logging evaluation metrics...
2026-06-04 19:32:32,211 - INFO - Saving and logging the model...
c:\Users\Admin\anaconda3\envs\atlas\lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditio

FINDING HYPERPARAMETERS 

In [15]:
import mlflow
import logging
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting Hyperparameter Tuning Parent Run...")

# Define the hyperparameter grid you want to search through
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],          # Regularization strength (smaller = stronger regularization)
    'penalty': ['l2'],                    # Use 'l2' (Ridge). If using 'l1', change solver to 'liblinear'
    'test_size': [0.20, 0.25, 0.30]       # Testing different data split ratios
}

best_accuracy = 0.0
best_params = {}
best_model = None

# 1. Start a parent run to group all tuning experiments together
with mlflow.start_run(run_name="Logistic_Regression_Tuning"):
    
    # Loop through your test splits
    for test_size in param_grid['test_size']:
        
        # Assuming you have your full dataset 'X' and 'y' loaded in memory
        # Dynamically split based on the current hyperparameter grid choice
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
        
        # Loop through model hyperparameter combinations
        for C in param_grid['C']:
            for penalty in param_grid['penalty']:
                
                # 2. Start a nested child run for this specific combination
                with mlflow.start_run(run_name=f"C_{C}_penalty_{penalty}_split_{test_size}", nested=True):
                    start_time = time.time()
                    
                    try:
                        # Log preprocessing and model parameters
                        mlflow.log_param("vectorizer", "Bag of Words")
                        mlflow.log_param("num_features", 1000)
                        mlflow.log_param("test_size", test_size)
                        mlflow.log_param("C", C)
                        mlflow.log_param("penalty", penalty)
                        mlflow.log_param("model", "Logistic Regression")

                        # Initialize and train model with current hyperparameters
                        model = LogisticRegression(C=C, penalty=penalty, max_iter=1000, solver='lbfgs')
                        model.fit(X_train, y_train)

                        # Predict and evaluate
                        y_pred = model.predict(X_test)
                        accuracy = accuracy_score(y_test, y_pred)
                        precision = precision_score(y_test, y_pred)
                        recall = recall_score(y_test, y_pred)
                        f1 = f1_score(y_test, y_pred)

                        # Log evaluation metrics for this run
                        mlflow.log_metric("accuracy", accuracy)
                        mlflow.log_metric("precision", precision)
                        mlflow.log_metric("recall", recall)
                        mlflow.log_metric("f1_score", f1)

                        # Explicitly define model signature to prevent MLflow lazy-loading import bugs
                        from mlflow.models import infer_signature
                        signature = infer_signature(X_test, y_pred)
                        mlflow.sklearn.log_model(model, "model", signature=signature)

                        # Track the absolute best configuration
                        if accuracy > best_accuracy:
                            best_accuracy = accuracy
                            best_model = model
                            best_params = {"C": C, "penalty": penalty, "test_size": test_size}

                        end_time = time.time()
                        logging.info(f"Finished combination C={C}, penalty={penalty}, split={test_size} | Accuracy: {accuracy:.4f} in {end_time - start_time:.2f}s")

                    except Exception as e:
                        logging.error(f"Failed on configuration C={C}, penalty={penalty}: {e}", exc_info=True)

    # 3. Log the overall best parameters to the parent run summary
    mlflow.log_param("best_C", best_params.get("C"))
    mlflow.log_param("best_penalty", best_params.get("penalty"))
    mlflow.log_param("best_test_size", best_params.get("test_size"))
    mlflow.log_metric("max_accuracy", best_accuracy)
    
    logging.info("--- Tuning Complete ---")
    logging.info(f"Highest Accuracy Achieved: {best_accuracy:.4f}")
    logging.info(f"Best Hyperparameters: {best_params}")


2026-06-04 19:21:30,341 - INFO - Starting Hyperparameter Tuning Parent Run...
c:\Users\Admin\anaconda3\envs\atlas\lib\site-packages\_distutils_hack\__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\Admin\anaconda3\envs\atlas\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
2026-06-04 19:21:44,976 - INFO - Finished combination C=0

KeyboardInterrupt: 